In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix


In [11]:
df = pd.read_csv("/content/final_user_guidance_with_stress.csv")

df.head()

,user_id,stress_score,stress_level,archetypes,rule_id,category,priority,message
0,1,0.6865,High,"Stressed_Professional, Money_Replanner, Ambiti...",R1,career,5,You are in a phase where focused effort on dee...
1,1,0.6865,High,"Stressed_Professional, Money_Replanner, Ambiti...",R2,career,4,Your appetite for calculated risk is an asset....
2,1,0.6865,High,"Stressed_Professional, Money_Replanner, Ambiti...",R5,health,4,High work pressure is not permanent but ignori...
3,1,0.6865,High,"Stressed_Professional, Money_Replanner, Ambiti...",R7,financial,4,Important money choices are around you. List y...
4,2,0.4844,Medium,Growth_Leader,R9,career,5,You have a concrete growth window. Clarify wha...


In [19]:
print(user_df.columns)


Index(['user_id', 'stress_score', 'stress_level', 'total_guidance',
       'avg_priority', 'health_guidance_count'],
      dtype='object')


In [14]:
user_df = (
    df
    .groupby("user_id")
    .agg(
        stress_score=("stress_score", "first"),
        stress_level=("stress_level", "first"),
        total_guidance=("rule_id", "count"),
        avg_priority=("priority", "mean"),
        health_guidance_count=("category", lambda x: (x == "health").sum())
    )
    .reset_index()
)

user_df.head()


,user_id,stress_score,stress_level,total_guidance,avg_priority,health_guidance_count
0,1,0.6865,High,4,4.25,1
1,2,0.4844,Medium,1,5.00,0
2,3,0.5732,Medium,1,5.00,1
3,4,0.5060,Medium,2,4.00,0
4,5,0.8750,High,2,4.50,0


In [20]:
user_df["high_stress"] = (user_df["stress_level"] == "High").astype(int)


In [21]:
user_df[["user_id", "stress_level", "high_stress"]].head()


,user_id,stress_level,high_stress
0,1,High,1
1,2,Medium,0
2,3,Medium,0
3,4,Medium,0
4,5,High,1


In [22]:
X = user_df[
    ["stress_score", "total_guidance", "avg_priority", "health_guidance_count"]
]

y = user_df["high_stress"]


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.67      1.00      0.80         2
           1       0.00      0.00      0.00         1

    accuracy                           0.67         3
   macro avg       0.33      0.50      0.40         3
weighted avg       0.44      0.67      0.53         3



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Load final rule-engine JSON (already includes stress)

In [27]:
import json

In [28]:
with open("/content/final_rule_engine_output_with_stress.json", "r") as f:
    users_json = json.load(f)

len(users_json)


10

In [29]:
rows = []

for user in users_json:
    rows.append({
        "user_id": int(user["user_id"]),
        "stress_score": user["stress_score"],
        "total_guidance": len(user["guidance"]),
        "avg_priority": np.mean([g["priority"] for g in user["guidance"]]) if user["guidance"] else 0,
        "health_guidance_count": sum(
            1 for g in user["guidance"] if g["category"] == "health"
        ),
        "high_stress": 1 if user["stress_level"] == "High" else 0
    })

user_df = pd.DataFrame(rows)
user_df.head()


,user_id,stress_score,total_guidance,avg_priority,health_guidance_count,high_stress
0,1,0.6865,4,4.25,1,1
1,2,0.4844,1,5.00,0,0
2,3,0.5732,1,5.00,1,0
3,4,0.5060,2,4.00,0,0
4,5,0.8750,2,4.50,0,1


In [30]:
feature_cols = [
    "stress_score",
    "total_guidance",
    "avg_priority",
    "health_guidance_count"
]

X = user_df[feature_cols]
y = user_df["high_stress"]

model = LogisticRegression()
model.fit(X, y)


LogisticRegression()

In [31]:
def run_inference(user_id: int):
    # ---- fetch user ----
    user = next(
        (u for u in users_json if int(u["user_id"]) == user_id),
        None
    )

    if user is None:
        return {"error": "User not found"}

    # ---- build feature row ----
    features = pd.DataFrame([{
        "stress_score": user["stress_score"],
        "total_guidance": len(user["guidance"]),
        "avg_priority": np.mean([g["priority"] for g in user["guidance"]]) if user["guidance"] else 0,
        "health_guidance_count": sum(
            1 for g in user["guidance"] if g["category"] == "health"
        )
    }])

    # ---- predict ----
    high_stress_pred = int(model.predict(features)[0])

    # ---- re-rank guidance ----
    guidance = user["guidance"].copy()

    if high_stress_pred == 1:
        # boost health guidance
        for g in guidance:
            if g["category"] == "health":
                g["priority"] += 2

    guidance = sorted(guidance, key=lambda x: x["priority"], reverse=True)

    # ---- final output ----
    return {
        "user_id": user_id,
        "predicted_high_stress": bool(high_stress_pred),
        "stress_score": user["stress_score"],
        "stress_level": user["stress_level"],
        "guidance": guidance
    }


In [32]:
for uid in [1, 3, 5]:
    print(json.dumps(run_inference(uid), indent=2))
    print("-" * 60)


{
  "user_id": 1,
  "predicted_high_stress": true,
  "stress_score": 0.6865,
  "stress_level": "High",
  "guidance": [
    {
      "rule_id": "R5",
      "category": "health",
      "priority": 6,
      "message": "High work pressure is not permanent but ignoring it can be costly. Block 15 minutes daily to decompress and review priorities."
    },
    {
      "rule_id": "R1",
      "category": "career",
      "priority": 5,
      "message": "You are in a phase where focused effort on deep work can pay off. Protect your calendar and say no to low value tasks."
    },
    {
      "rule_id": "R2",
      "category": "career",
      "priority": 4,
      "message": "Your appetite for calculated risk is an asset. Explore one bold move in the next 30 days rather than many small scattered ones."
    },
    {
      "rule_id": "R7",
      "category": "financial",
      "priority": 4,
      "message": "Important money choices are around you. List your top two long term goals and check if this deci

In [34]:
import joblib
joblib.dump(model, "AI_Life_Guidance_Engine.pkl")


['AI_Life_Guidance_Engine.pkl']